# create_agent() 函数

## LangChain create\_agent() 函数

create\_agent() 是 LangChain 最核心的函数，它会创建一个完整的 Agent 图（StateGraph），包含模型调用、工具执行、循环控制等全部逻辑。

### 语法

create\_agent() 函数语法格式如下：

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model,                     # str | BaseChatModel：语言模型
    tools=None,                # Sequence：工具列表
    *,
    system_prompt=None,        # str | SystemMessage：系统提示
    middleware=(),             # Sequence[AgentMiddleware]：中间件列表
    response_format=None,      # ResponseFormat | type：结构化输出配置
    state_schema=None,         # type[AgentState]：自定义状态结构
    context_schema=None,       # type：运行时上下文结构
    checkpointer=None,         # Checkpointer：对话持久化
    store=None,                # BaseStore：跨会话存储
    interrupt_before=None,     # list[str]：在哪些节点前暂停
    interrupt_after=None,      # list[str]：在哪些节点后暂停
    debug=False,               # bool：是否输出详细日志
    name=None,                 # str：Agent 名称
    cache=None,                # BaseCache：缓存配置
)

### model 参数——模型配置

model 可以接受两种形式：字符串（由 init\_chat\_model() 处理）或已构建好的 BaseChatModel 实例。

## 实例

from langchain.agents import create\_agent  
from langchain.chat\_models import init\_chat\_model  
  
\# 方式 1：传字符串（最常用）  
\# create\_agent 内部会调用 init\_chat\_model() 处理  
agent = create\_agent(  
model="deepseek:deepseek-v4-flash",  
system\_prompt="你是菜鸟教程 RUNOOB 的助手",  
)  
  
\# 方式 2：传已构建好的模型实例  
\# 适合需要精细控制模型参数的场景  
model = init\_chat\_model("deepseek:deepseek-v4-flash", temperature=0.3, max\_tokens=500)  
agent = create\_agent(  
model=model,  
system\_prompt="你是菜鸟教程 RUNOOB 的助手",  
)  
  
\# 方式 3：传已绑定工具的模型实例  
\# less common，通常让 create\_agent 自己管理工具绑定  
model\_with\_tools = init\_chat\_model("deepseek:deepseek-v4-flash").bind\_tools(\[...\])

> 推荐使用方式 1（传字符串）。create\_agent() 内部会自动处理模型初始化、工具绑定、结构化输出等逻辑。方式 2 适合你需要在 Agent 之外也使用同一个模型实例的场景。

### tools 参数——工具列表

tools 接受三种格式的工具：

## 实例

from langchain.tools import tool  
from langchain.agents import create\_agent  
  
\# 格式 1：@tool 装饰的函数（最常用）  
@tool  
def search\_course(keyword: str) -> str:  
"""搜索菜鸟教程课程"""  
return f"搜索结果：{keyword} 相关课程"  
  
\# 格式 2：Pydantic BaseModel 类  
from pydantic import BaseModel, Field  
  
class WeatherQuery(BaseModel):  
"""查询天气"""  
city: str = Field(description="城市名称")  
  
\# 格式 3：字典（描述远程工具或内置工具）  
mcp\_tool = {  
"type": "mcp",  
"server\_label": "weather\_server",  
"server\_url": "https://weather.example.com/sse",  
"allowed\_tools": \["get\_forecast"\],  
}  
  
\# 混合使用  
agent = create\_agent(  
model="deepseek:deepseek-v4-flash",  
tools=\[search\_course, WeatherQuery, mcp\_tool\],  
)

传 None 或空列表表示 Agent 无工具可用，此时它就是一个纯粹的对话模型：

## 实例

from langchain.agents import create\_agent  
from langchain.messages import HumanMessage  
  
\# 无工具 Agent——等价于直接调用模型  
agent = create\_agent(  
model="deepseek:deepseek-v4-flash",  
tools=None,  
system\_prompt="你是菜鸟教程 RUNOOB 的助手",  
)  
  
result = agent.invoke({  
"messages": \[HumanMessage(content="Python 适合零基础学习吗？")\]  
})  
print(result\["messages"\]\[-1\].content)

### system\_prompt 参数——系统提示

定义 Agent 的行为角色和约束规则。支持字符串和 SystemMessage 对象。

## 实例

from langchain.agents import create\_agent  
from langchain.messages import SystemMessage  
  
\# 方式 1：字符串（简单直接）  
agent = create\_agent(  
model="deepseek:deepseek-v4-flash",  
system\_prompt="你是菜鸟教程 RUNOOB 的学习顾问。回答要简洁，不超过 100 字。",  
)  
  
\# 方式 2：SystemMessage 对象（可在多个 Agent 间复用）  
system\_msg = SystemMessage(  
content="你是菜鸟教程 RUNOOB 的学习顾问。回答要简洁，不超过 100 字。"  
)  
agent = create\_agent(  
model="deepseek:deepseek-v4-flash",  
system\_prompt=system\_msg,  
)

> system\_prompt 是可选的，但不传的话模型会以"通用助手"的角色回答。对于有明确业务场景的应用，建议始终设置 system\_prompt 来约束模型的行为边界。

### state\_schema 参数——自定义状态

默认的 AgentState 只包含 messages、jump\_to 和 structured\_response。如果你需要额外的状态字段，可以扩展它：

## 实例

from typing import Annotated  
from langchain.agents import create\_agent, AgentState  
from langchain.messages import HumanMessage  
from langchain.tools import tool, InjectedState  
from typing\_extensions import TypedDict  
  
\# 扩展 AgentState，添加自定义字段  
class LearningAgentState(AgentState):  
"""自定义状态，增加学习进度相关字段"""  
user\_level: str # 用户等级  
completed\_topics: list\[str\] # 已完成的主题列表  
  
@tool  
def track\_progress(  
topic: str,  
state: Annotated\[dict, InjectedState\],  
) -> str:  
"""记录用户的学习进度。  
  
Args:  
topic: 刚学完的主题名称  
"""  
completed = state.get("completed\_topics", \[\])  
completed.append(topic)  
return (  
f"已记录学习进度。当前已完成 {len(completed)} 个主题："  
f"{', '.join(completed)}"  
)  
  
agent = create\_agent(  
model="deepseek:deepseek-v4-flash",  
tools=\[track\_progress\],  
state\_schema=LearningAgentState, # 使用自定义状态  
system\_prompt="你是菜鸟教程 RUNOOB 的学习助手。",  
)  
  
\# 运行时需要提供自定义状态的初始值  
result = agent.invoke({  
"messages": \[HumanMessage(content="我学完了 Python 基础，帮我记录一下")\],  
"user\_level": "入门",  
"completed\_topics": \["HTML 基础"\],  
})  
  
print(f"用户等级: {result.get('user\_level')}")  
print(f"已完成主题: {result.get('completed\_topics')}")  
print(f"回复: {result\['messages'\]\[-1\].content\[:100\]}")

运行结果：

In [ ]:
用户等级: 入门
已完成主题: ['HTML 基础', 'Python 基础']
回复: 已记录学习进度。当前已完成 2 个主题：HTML 基础, Python 基础

### 返回值——CompiledStateGraph

create\_agent() 返回一个 CompiledStateGraph 对象，这是 LangGraph 的编译后的图，提供了多种运行方式：

| 方法 | 说明 | 适用场景 |
| --- | --- | --- |
| invoke(input, config) | 同步运行，等待完整结果 | 脚本、简单接口 |
| ainvoke(input, config) | 异步运行，等待完整结果 | Web 服务 |
| stream(input, config, stream\_mode) | 同步流式运行 | 实时展示中间步骤 |
| astream(input, config, stream\_mode) | 异步流式运行 | WebSocket、SSE |
| get\_state(config) | 获取当前状态 | 查看/恢复对话状态 |
| update\_state(config, values) | 更新状态 | 手动修改对话状态 |